<a href="https://colab.research.google.com/github/Ashishkumar07777/ML-PROJECTS/blob/main/Customer_Churn_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. **IMPORTING THE DEPENDENCIES**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report


2. **DATA LOADING AND UNDERSTANDING**

In [2]:
df=pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
pd.set_option('display.max_columns',None)

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
df=df.drop(columns=["customerID"])

In [ ]:
df.head(2)

In [ ]:
df.columns

In [ ]:
print(df["gender"].unique())

In [ ]:
print(df["SeniorCitizen"].unique())

In [ ]:
numerical_features_list=["tenure","MonthlyCharges","TotalCharges"]

for col in df.columns:
  if col not in numerical_features_list:
    print(col,df[col].unique())
    print("-"*50)

In [ ]:
print(df.isnull().sum())

In [ ]:
df[df["TotalCharges"]==" "]

In [ ]:
len(df[df["TotalCharges"]==" "])

In [ ]:
df["TotalCharges"]=df["TotalCharges"].replace({" ": "0.0"})

In [ ]:
df["TotalCharges"]=df["TotalCharges"].astype(float)

In [ ]:
df.info()

In [ ]:
print(df["Churn"].value_counts())

**Insights:**

1. Customer ID removed as it is not required for modelling
2. No mmissing values in the dataset
3. Missing values in the TotalCharges column were replaced with 0
4. Class imbalance identified in the target

3. EXPLORATORY DATA ANALYSIS(EDA)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
df.describe()

**NUMERICAL FEATURES-ANALYSIS**

* Understand the distribution of tech numerical features

In [ ]:
def plot_histogram(df,column_name):
  plt.figure(figsize=(5,3))
  sns.histplot(df[column_name],kde=True)
  plt.title(f"Distribution of {column_name}")

  col_mean =df[column_name].mean()
  col_median=df[column_name].median()

  plt.axvline(col_mean,color="red",linestyle="--",label="Mean")
  plt.axvline(col_median,color="green",linestyle="-",label="Median")

  plt.legend()
  plt.show()


In [ ]:
plot_histogram(df,"tenure")

In [ ]:
plot_histogram(df,"MonthlyCharges")

In [ ]:
plot_histogram(df, "TotalCharges")

BOX PLOT NUMERICAL FEATURES

In [ ]:
def plot_box

In [ ]:
def plot_boxplot(df, column_name):
  plt.figure(figsize=(5, 3))
  sns.boxplot(y=df[column_name])
  plt.title(f"Box plot of {column_name}")
  plt.ylabel(column_name)
  plt.show()

plot_boxplot(df, "tenure")

In [ ]:
plot_boxplot(df, "MonthlyCharges")

In [ ]:
plot_boxplot(df, "TotalCharges")

COORELATION HEATMAP FOR NUMERICAL COLUMNS

In [ ]:
plt.figure(figsize=(8,4))
sns.heatmap(df[["tenure","MonthlyCharges","TotalCharges"]].corr(),annot=True,cmap="coolwarm",fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

CATEGORICAL FEATURES-ANALYSIS

In [ ]:
df.columns

In [ ]:
df.info()

COUNTPLOT FOR CATEGORICAL COLUMNS

In [ ]:
object_cols=df.select_dtypes(include="object").columns.to_list()
object_cols=["SeniorCitizen"] + object_cols

for col in object_cols:
  plt.figure(figsize=(5,3))
  sns.countplot(x=df[col])
  plt.title(f"Count Plot of {col}")
  plt.show()

In [ ]:
df.head(3)

LABEL ENCODING OF TARGET COLUMN

In [ ]:
df["Churn"] = df["Churn"].replace({"Yes":1,"No":0})

In [ ]:
print(df["Churn"].value_counts())

LABEL ENCODING OF CATEGROICAL FEATURES

In [ ]:
object_columns=df.select_dtypes(include="object").columns

In [ ]:
print(object_columns)

In [ ]:
encoder={}
for column in object_columns:
  label_encoder = LabelEncoder()
  df[column] = label_encoder.fit_transform(df[column])
  encoder[column] = label_encoder

  with open("encoder.pkl","wb")as f:
    pickle.dump(encoder,f)

In [ ]:
encoder

In [ ]:
df.head()

TRAINING AND TEST DATA SPLIT

In [ ]:
x=df.drop(columns=["Churn"])
y=df["Churn"]

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
print(y_train.shape)

In [ ]:
print(y_train.value_counts())

SYNTHETIC MINORITY OVERSAMPLING TECHNIQUE(SMOTE)

In [ ]:
smote=SMOTE(random_state=42)

In [ ]:
x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)

In [ ]:
print(y_train_smote.shape)

In [ ]:
print(y_train_smote.value_counts())

5.MODEL TRAINING

TRAINING WITH DEFAULT HYPERPARAMETERS

In [ ]:
models={
    "Decision Tree":DecisionTreeClassifier(),
    "Random Forest":RandomForestClassifier(),
    "XGBoost":XGBClassifier()
}

In [ ]:

cv_scores = {}


for model_name, model in models.items():
  print(f"Training {model_name} with default parameters")
  scores = cross_val_score(model, x_train_smote, y_train_smote, cv=5, scoring="accuracy")
  cv_scores[model_name] = scores
  print(f"{model_name} cross-validation accuracy: {np.mean(scores):.2f}")
  print("-"*70)

In [ ]:
cv_scores

RANDOM FOREST GIVES THE HIGHEST ACCURACY COMPARED TO OTHER MODELS WITH DEAFAULT PARAMETERS

In [ ]:
rfc=RandomForestClassifier(random_state=42)

In [ ]:
rfc.fit(x_train_smote,y_train_smote)

In [ ]:
print(y_test.value_counts())

6.MODEL EVALUATION

In [ ]:
y_test_pred = rfc.predict(x_test)

print("Accuracy Score:\n", accuracy_score(y_test, y_test_pred))
print("Confsuion Matrix:\n", confusion_matrix(y_test, y_test_pred))
print("Classification Report:\n", classification_report(y_test, y_test_pred))

In [ ]:
model_data={"model": rfc,"features_names":x.columns.tolist()}
with open("customer_churn_model.pkl","wb") as f:
  pickle.dump(model_data,f)

7. LOAD THE SAVED MODEL AND BUILD A PREDICTIVE SYSTEM

In [ ]:
with open("customer_churn_model.pkl", "rb") as f:
  model_data = pickle.load(f)

loaded_model = model_data["model"]
feature_names = model_data["features_names"]

In [ ]:
print(loaded_model)

In [ ]:
print(feature_names)

In [ ]:
input_data = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 1,
    'PhoneService': 'No',
    'MultipleLines': 'No phone service',
    'InternetService': 'DSL',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'Yes',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'No',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 29.85,
    'TotalCharges': 29.85,
    'Totalcharges': 29.85
}


input_data_df = pd.DataFrame([input_data])

with open("encoder.pkl", "rb") as f:
  encoders = pickle.load(f)



for column, encoder in encoders.items():
  if column in input_data_df.columns:
    input_data_df[column] = encoder.transform(input_data_df[column])

.
input_data_df = input_data_df[feature_names]


prediction = loaded_model.predict(input_data_df)
pred_prob = loaded_model.predict_proba(input_data_df)

print(prediction)


print(f"Prediction: {'Churn' if prediction[0] == 1 else 'No Churn'}")
print(f"Prediciton Probability: {pred_prob}")

In [ ]:
encoders

1. Implement Hyperparameter Tuining
2. Try Model Selection
3. Try downsampling
4. Try to address teh overfitting
5. Try Startified k fold CV

## Define Parameter Grid



In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [10, 20, 30, None],
    'criterion': ['gini', 'entropy'],
    'random_state': [42]
}
print("Hyperparameter grid for Random Forest Classifier defined:")
print(param_grid)

## Perform GridSearchCV




In [ ]:
from sklearn.model_selection import GridSearchCV

rfc = RandomForestClassifier(random_state=42)


grid_search = GridSearchCV(estimator=rfc,
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1,
                           verbose=2)


grid_search.fit(x_train_smote, y_train_smote)

print("GridSearchCV completed.")

In [ ]:
print("Best hyperparameters found:", grid_search.best_params_)
print("Best accuracy score:", grid_search.best_score_)

In [ ]:
best_rfc = RandomForestClassifier(**grid_search.best_params_)
best_rfc.fit(x_train_smote, y_train_smote)
y_pred_tuned = best_rfc.predict(x_test)

print("Model training with best parameters and prediction on test set completed.")

In [ ]:
print("Accuracy Score (Tuned Model):\n", accuracy_score(y_test, y_pred_tuned))
print("Confusion Matrix (Tuned Model):\n", confusion_matrix(y_test, y_pred_tuned))
print("Classification Report (Tuned Model):\n", classification_report(y_test, y_pred_tuned))

## Implement Random Downsampling




In [ ]:
from imblearn.under_sampling import RandomUnderSampler


rus = RandomUnderSampler(random_state=42)


x_train_downsampled, y_train_downsampled = rus.fit_resample(x_train, y_train)


print("Value counts of y_train_downsampled:")
print(y_train_downsampled.value_counts())

**Reasoning**:
Now that the training data has been downsampled, the next step is to evaluate the performance of Decision Tree, Random Forest, and XGBoost models using `StratifiedKFold` cross-validation on this newly balanced dataset. This will provide a comparative analysis of their accuracy scores on the downsampled data.



In [ ]:
from sklearn.model_selection import StratifiedKFold

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_downsampled_scores = {}


for model_name, model in models.items():
  print(f"Performing StratifiedKFold cross-validation for {model_name} on downsampled data")
  scores = cross_val_score(model, x_train_downsampled, y_train_downsampled, cv=skf, scoring="accuracy")
  cv_downsampled_scores[model_name] = scores
  print(f"{model_name} cross-validation accuracy on downsampled data: {np.mean(scores):.2f}")
  print("-"*70)

print("Cross-validation on downsampled data completed.")

## Evaluate Models with Downsampled Data using Stratified K-Fold Cross-Validation




In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Initialize all models, including the new ones
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, solver='liblinear'), # solver 'liblinear' is good for small datasets and binary classification
    "Support Vector Machine": SVC(gamma='auto', random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

# Initialize StratifiedKFold for cross-validation (already initialized as skf in previous steps, but re-initializing for clarity here)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_downsampled_scores_all = {}

# Perform 5-fold stratified cross-validation for each model on downsampled data
for model_name, model in models.items():
  print(f"Performing StratifiedKFold cross-validation for {model_name} on downsampled data")
  scores = cross_val_score(model, x_train_downsampled, y_train_downsampled, cv=skf, scoring="accuracy")
  cv_downsampled_scores_all[model_name] = scores
  print(f"{model_name} cross-validation accuracy on downsampled data: {np.mean(scores):.2f}")
  print("-"*70)

print("Cross-validation on downsampled data with all models completed.")

## Compare Model Performance

### Subtask:
Compare the performance of all models evaluated with downsampling and stratified k-fold cross-validation, focusing on metrics such as accuracy, precision, recall, and F1-score for both classes. This will help in selecting the 'best' model.


**Reasoning**:
To compare the models, I will calculate and print the average cross-validation accuracy for each model, then identify the model with the highest average accuracy to determine the best performer.



In [ ]:
print("\n--- Model Comparison on Downsampled Data ---")

best_model_name = ""
best_avg_accuracy = 0

for model_name, scores in cv_downsampled_scores_all.items():
  avg_accuracy = np.mean(scores)
  print(f"{model_name}: Average Cross-Validation Accuracy = {avg_accuracy:.4f}")

  if avg_accuracy > best_avg_accuracy:
    best_avg_accuracy = avg_accuracy
    best_model_name = model_name

print("\n--- Best Performing Model ---")
print(f"The best performing model based on average cross-validation accuracy is: {best_model_name} with an average accuracy of {best_avg_accuracy:.4f}")

**Reasoning**:
The previous steps have already compared models based on average cross-validation accuracy and identified the best performer. To fulfill the subtask's requirement of focusing on precision, recall, and F1-score for both classes, I need to train the best model (Logistic Regression) on the entire downsampled training data and then evaluate its performance on the untouched test set using a classification report.



In [ ]:
rom sklearn.metrics import classification_report


best_model = LogisticRegression(random_state=42, solver='liblinear')
best_model.fit(x_train_downsampled, y_train_downsampled)


y_pred_best_model = best_model.predict(x_test)


print(f"--- Performance of {best_model_name} on Test Set ---")
print("Accuracy Score:", accuracy_score(y_test, y_pred_best_model))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best_model))
print("Classification Report:\n", classification_report(y_test, y_pred_best_model))